# Tests de mini-funcionalidades de OP-11 `read_flows`

Este notebook se usa para probar helpers y bloques internos de `read_flows()` antes de hacer smoke tests o tests integrados de la función pública completa.

Objetivo:

- verificar minifuncionalidades de lectura de forma aislada;
- cubrir la resolución formal del bundle `.golondrina`;
- probar carga, recuperación y validación de sidecar;
- comprobar lectura backend-aware de Parquet y Feather;
- revisar la política opcional de `flow_to_trips`;
- dejar una base fácil de portar después a `pytest`.

Convenciones:

- los tests usan `assert`;
- se prueban helpers internos de OP-11, no la función pública completa;
- no se incluyen todavía smoke tests;
- no se prueban helpers de `write_flows`, porque esos corresponden a OP-10.

## Bloque 1. Preparación

### 1.1 Imports generales

Qué prepara: imports básicos, utilidades de filesystem, escritura mínima de artefactos Parquet/Feather y estructuras usadas en los tests.

In [1]:
import copy
import json
import shutil
from pathlib import Path

import pandas as pd
import pyarrow.feather as feather

### 1.2 Imports del módulo

Qué prepara: imports de opciones, excepciones y helpers reales usados por OP-11 `read_flows`.

In [2]:
from pylondrina.errors import ExportError

from pylondrina.io.flows import (
    ReadFlowsOptions,

    _validate_read_layout,
    _load_flow_sidecar,
    _recover_flow_read_state,
    _read_flows_table,
    _read_optional_flow_to_trips,
    _build_read_summary,

    _resolve_flows_artifact_root_for_read,
    _resolve_flows_artifact_paths,
    _resolve_flows_data_path_from_sidecar,
    _resolve_flow_to_trips_path_from_sidecar,

    _flow_data_filename_for_storage,
    _flow_to_trips_filename_for_storage,

    _options_to_read_parameters,
)

### 1.3 Helpers de apoyo para test

Qué prepara: utilidades pequeñas para assertions, inspección de issues y validación de payloads serializables.

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_dumpable(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [
        issue.code if hasattr(issue, "code") else issue.get("code")
        for issue in issues
    ]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, (
        f"No se encontró el issue {code}. "
        f"Codes actuales: {codes}"
    )


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, (
        f"Se encontró inesperadamente el issue {code}. "
        f"Codes actuales: {codes}"
    )

### 1.4 Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## Bloque 2. Fixtures reutilizables mínimas

Qué prepara:

- una carpeta local `./tmp_helper_read_flows` para artefactos de prueba;
- factories de tablas `flows` y `flow_to_trips`;
- sidecars válidos Parquet/Feather;
- un helper para materializar bundles formales mínimos sin usar todavía la función pública `read_flows`.

In [5]:
HELPER_ROOT = Path("./tmp_helper_read_flows")


def reset_helper_root() -> Path:
    if HELPER_ROOT.exists():
        shutil.rmtree(HELPER_ROOT)
    HELPER_ROOT.mkdir(parents=True, exist_ok=True)
    return HELPER_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = HELPER_ROOT / case_name
    if case_dir.exists():
        shutil.rmtree(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def make_flows_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "flow_id": ["f1", "f2", "f3"],
            "origin_h3_index": [
                "881111111111111",
                "882222222222222",
                "883333333333333",
            ],
            "destination_h3_index": [
                "884444444444444",
                "885555555555555",
                "886666666666666",
            ],
            "flow_count": [2, 1, 3],
            "flow_value": [2.0, 1.0, 4.5],
            "mode": ["bus", "metro", "bus"],
            "window_start_utc": [
                "2026-01-01T08:00:00Z",
                "2026-01-01T09:00:00Z",
                "2026-01-01T10:00:00Z",
            ],
            "window_end_utc": [
                "2026-01-01T08:59:59Z",
                "2026-01-01T09:59:59Z",
                "2026-01-01T10:59:59Z",
            ],
        }
    )


def make_flow_to_trips_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "flow_id": ["f1", "f1", "f2"],
            "movement_id": ["m1", "m2", "m3"],
        }
    )


def write_df_with_backend(
    df: pd.DataFrame,
    path: Path,
    *,
    storage_format: str,
):
    if storage_format == "parquet":
        df.to_parquet(path, index=False)
        return

    if storage_format == "feather":
        df.reset_index(drop=True).to_feather(path)
        return

    raise ValueError(f"storage_format inesperado: {storage_format!r}")


def make_sidecar_payload(
    *,
    storage_format: str = "feather",
    include_flow_to_trips: bool = True,
) -> dict:
    if storage_format == "parquet":
        data_name = "flows.parquet"
        aux_name = "flow_to_trips.parquet"
        storage_options = {"compression": "snappy"}
    elif storage_format == "feather":
        data_name = "flows.feather"
        aux_name = "flow_to_trips.feather"
        storage_options = {"compression": "lz4", "version": 2}
    else:
        raise ValueError(f"storage_format inesperado: {storage_format!r}")

    payload = {
        "dataset_type": "flows",
        "format": "golondrina",
        "layout_version": "1.1",
        "storage": {
            "format": storage_format,
            "options": storage_options,
        },
        "dataset_id": "dset_sidecar",
        "artifact_id": "art_sidecar",
        "files": {
            "data": data_name,
            "metadata": "flows.metadata.json",
            "flow_to_trips": (
                aux_name if include_flow_to_trips else None
            ),
        },
        "aggregation_spec": {
            "h3_resolution": 8,
            "group_by": ["mode"],
            "time_aggregation": "hour",
            "time_basis": "origin",
            "min_trips_per_flow": 1,
        },
        "provenance": {
            "derived_from": [
                {
                    "type": "trips",
                    "dataset_id": "trip_dset_001",
                }
            ],
        },
        "metadata": {
            "dataset_id": "dset_sidecar",
            "artifact_id": "art_sidecar",
            "is_validated": True,
            "events": [],
        },
        "tables": {
            "flows": {
                "n_rows": len(make_flows_df()),
                "n_cols": len(make_flows_df().columns),
                "columns": list(make_flows_df().columns),
            },
            "flow_to_trips": {
                "n_rows": len(make_flow_to_trips_df()),
                "n_cols": len(make_flow_to_trips_df().columns),
                "columns": list(make_flow_to_trips_df().columns),
            } if include_flow_to_trips else None,
        },
    }
    return payload


def materialize_minimal_formal_flow_artifact(
    root: Path,
    *,
    storage_format: str = "feather",
    with_aux: bool = True,
):
    root.mkdir(parents=True, exist_ok=True)

    paths = _resolve_flows_artifact_paths(root)

    data_filename = _flow_data_filename_for_storage(storage_format)
    aux_filename = _flow_to_trips_filename_for_storage(storage_format)

    data_path = root / data_filename
    aux_path = root / aux_filename

    write_df_with_backend(
        make_flows_df(),
        data_path,
        storage_format=storage_format,
    )

    if with_aux:
        write_df_with_backend(
            make_flow_to_trips_df(),
            aux_path,
            storage_format=storage_format,
        )

    payload = make_sidecar_payload(
        storage_format=storage_format,
        include_flow_to_trips=with_aux,
    )

    paths.sidecar_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    return {
        "paths": paths,
        "payload": payload,
        "data_path": data_path,
        "aux_path": aux_path,
    }


root = reset_helper_root()
print("HELPER_ROOT =", root.resolve())
show_ok("Bloque 2 - fixtures mínimas para OP-11 read_flows")

HELPER_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_helper_read_flows
OK - Bloque 2 - fixtures mínimas para OP-11 read_flows


## Bloque 3. Helpers de root, paths y parameters

Este bloque prueba helpers previos a la lectura efectiva:

- resolución del root de lectura con fallback `.golondrina`;
- rutas base de `FlowsArtifactPaths`;
- serialización de `ReadFlowsOptions` en `parameters`;
- validación del layout formal mínimo.

### Test 3.1 - `_resolve_flows_artifact_root_for_read`

Qué prueba:

- si el path exacto existe, se usa tal cual;
- si no existe y existe la variante con `.golondrina`, se usa el fallback;
- si no existe ninguna variante, se conserva el path original para que el precheck posterior falle de manera controlada.

In [6]:
case_dir = make_case_dir("case_03_01_resolve_root")

exact_root = case_dir / "exact_bundle.golondrina"
exact_root.mkdir(parents=True, exist_ok=True)

resolved_exact = _resolve_flows_artifact_root_for_read(exact_root)
assert resolved_exact == exact_root


fallback_base = case_dir / "bundle_without_suffix"
fallback_true = case_dir / "bundle_without_suffix.golondrina"
fallback_true.mkdir(parents=True, exist_ok=True)

resolved_fallback = _resolve_flows_artifact_root_for_read(fallback_base)
assert resolved_fallback == fallback_true


missing_root = case_dir / "missing_bundle"
resolved_missing = _resolve_flows_artifact_root_for_read(missing_root)
assert resolved_missing == missing_root

show_ok("Test 3.1 - _resolve_flows_artifact_root_for_read")

OK - Test 3.1 - _resolve_flows_artifact_root_for_read


### Test 3.2 - `_resolve_flows_artifact_paths`

Qué prueba:

- el helper actual solo resuelve `root_dir` y `sidecar_path`;
- ya no expone `data_path` ni `flow_to_trips_path`, porque esas rutas dependen del backend declarado en el sidecar.

In [7]:
root = Path("tmp_demo_read_flows_artifact")
paths = _resolve_flows_artifact_paths(root)

assert paths.root_dir == root
assert paths.sidecar_path == root / "flows.metadata.json"

assert not hasattr(paths, "data_path")
assert not hasattr(paths, "flow_to_trips_path")

show_ok("Test 3.2 - _resolve_flows_artifact_paths")

OK - Test 3.2 - _resolve_flows_artifact_paths


### Test 3.3 - `_options_to_read_parameters`

Qué prueba:

- los parámetros efectivos de `read_flows` quedan serializados en forma estable;
- el path se expresa como string compatible con el sistema operativo actual;
- se preservan `strict`, `keep_metadata` y `read_flow_to_trips`.

In [8]:
options = ReadFlowsOptions(
    strict=True,
    keep_metadata=False,
    read_flow_to_trips=False,
)

input_path = Path("/tmp/demo_read_flows.golondrina")

parameters = _options_to_read_parameters(
    path=input_path,
    options=options,
)

assert parameters["path"] == str(input_path.expanduser())
assert parameters["strict"] is True
assert parameters["keep_metadata"] is False
assert parameters["read_flow_to_trips"] is False

assert_json_dumpable(parameters, "read_parameters")

show_ok("Test 3.3 - _options_to_read_parameters")

OK - Test 3.3 - _options_to_read_parameters


### Test 3.4 - `_validate_read_layout` happy path

Qué prueba:

- un root existente como directorio;
- sidecar `flows.metadata.json` presente;
- no se emiten issues.

In [9]:
case_dir = make_case_dir("case_03_04_validate_layout_happy")
root = case_dir / "artifact.golondrina"
root.mkdir(parents=True, exist_ok=True)

paths = _resolve_flows_artifact_paths(root)
paths.sidecar_path.touch()

issues = []

_validate_read_layout(
    root,
    paths,
    strict=False,
    issues=issues,
)

assert issues == []

show_ok("Test 3.4 - _validate_read_layout happy path")

OK - Test 3.4 - _validate_read_layout happy path


### Test 3.5 - `_validate_read_layout` fatal por root inválido

Qué prueba: el helper aborta cuando el root no existe o no es directorio.

In [10]:
case_dir = make_case_dir("case_03_05_invalid_root")
root = case_dir / "missing_artifact.golondrina"

paths = _resolve_flows_artifact_paths(root)
issues = []

try:
    _validate_read_layout(
        root,
        paths,
        strict=False,
        issues=issues,
    )
    raise AssertionError("Debió fallar por root inválido")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.PATH.INVALID_ROOT",
    )

show_ok("Test 3.5 - _validate_read_layout invalid root")

OK - Test 3.5 - _validate_read_layout invalid root


### Test 3.6 - `_validate_read_layout` fatal por sidecar faltante

Qué prueba: `flows.metadata.json` ausente no es recuperable en lectura formal.

In [11]:
case_dir = make_case_dir("case_03_06_missing_sidecar")
root = case_dir / "artifact.golondrina"
root.mkdir(parents=True, exist_ok=True)

paths = _resolve_flows_artifact_paths(root)
issues = []

try:
    _validate_read_layout(
        root,
        paths,
        strict=False,
        issues=issues,
    )
    raise AssertionError("Debió fallar por sidecar faltante")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.LAYOUT.MISSING_SIDECAR",
    )

show_ok("Test 3.6 - _validate_read_layout missing sidecar")

OK - Test 3.6 - _validate_read_layout missing sidecar


## Bloque 4. Carga de sidecar y recuperación de estado

Este bloque prueba dos responsabilidades centrales de OP-11:

- leer y validar el sidecar formal;
- recuperar el estado persistido de lectura antes de reconstruir el `FlowDataset`.

La recuperación incluye:

- backend efectivo;
- `dataset_id`;
- `artifact_id`;
- `aggregation_spec`;
- `provenance`;
- `metadata`;
- degradaciones controladas bajo `strict=False`.

### Test 4.1 - `_load_flow_sidecar` happy path

Qué prueba:

- sidecar JSON válido;
- top-level formal completo;
- retorno de un mapping usable;
- ausencia de issues.

In [12]:
case_dir = make_case_dir("case_04_01_load_sidecar_happy")
sidecar_path = case_dir / "flows.metadata.json"

payload = make_sidecar_payload(
    storage_format="feather",
    include_flow_to_trips=True,
)

sidecar_path.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

issues = []

loaded = _load_flow_sidecar(
    sidecar_path,
    strict=False,
    issues=issues,
    destination_path=case_dir,
)

assert loaded["dataset_type"] == "flows"
assert loaded["storage"]["format"] == "feather"
assert loaded["files"]["data"] == "flows.feather"
assert loaded["files"]["flow_to_trips"] == "flow_to_trips.feather"
assert issues == []

show_ok("Test 4.1 - _load_flow_sidecar happy path")

OK - Test 4.1 - _load_flow_sidecar happy path


### Test 4.2 - `_load_flow_sidecar` fatal por JSON ilegible

Qué prueba: sidecar no parseable aborta con issue de IO.

In [13]:
case_dir = make_case_dir("case_04_02_load_sidecar_invalid_json")
sidecar_path = case_dir / "flows.metadata.json"

sidecar_path.write_text(
    "{ invalid json ",
    encoding="utf-8",
)

issues = []

try:
    _load_flow_sidecar(
        sidecar_path,
        strict=False,
        issues=issues,
        destination_path=case_dir,
    )
    raise AssertionError("Debió fallar por JSON inválido")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.IO.SIDECAR_READ_FAILED",
    )

show_ok("Test 4.2 - _load_flow_sidecar invalid JSON")

OK - Test 4.2 - _load_flow_sidecar invalid JSON


### Test 4.3 - `_load_flow_sidecar` fatal por top-level incompleto

Qué prueba: el sidecar formal exige todas las claves top-level mínimas.

In [14]:
case_dir = make_case_dir("case_04_03_load_sidecar_bad_top_level")
sidecar_path = case_dir / "flows.metadata.json"

bad_payload = {
    "dataset_type": "flows",
    "format": "golondrina",
    # faltan varias claves obligatorias
}

sidecar_path.write_text(
    json.dumps(bad_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

issues = []

try:
    _load_flow_sidecar(
        sidecar_path,
        strict=False,
        issues=issues,
        destination_path=case_dir,
    )
    raise AssertionError("Debió fallar por top-level incompleto")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.SIDECAR.INVALID_TOP_LEVEL",
    )

show_ok("Test 4.3 - _load_flow_sidecar invalid top-level")

OK - Test 4.3 - _load_flow_sidecar invalid top-level


### Test 4.4 - `_recover_flow_read_state` normal para Parquet y Feather

Qué prueba:

- el backend se recupera desde `storage.format`;
- ids válidos se preservan;
- `aggregation_spec`, `provenance` y `metadata` se cargan sin degradación;
- no se emiten issues.

In [15]:
for storage_format in ["parquet", "feather"]:
    payload = make_sidecar_payload(
        storage_format=storage_format,
        include_flow_to_trips=True,
    )

    issues = []

    state = _recover_flow_read_state(
        payload,
        strict=False,
        issues=issues,
        destination_path=Path(f"/tmp/fake_{storage_format}.golondrina"),
    )

    assert state["storage_format"] == storage_format
    assert state["dataset_id"] == "dset_sidecar"
    assert state["artifact_id"] == "art_sidecar"
    assert state["aggregation_spec"]["h3_resolution"] == 8
    assert state["provenance"]["derived_from"][0]["type"] == "trips"
    assert state["metadata"]["dataset_id"] == "dset_sidecar"
    assert issues == []

show_ok("Test 4.4 - _recover_flow_read_state normal parquet/feather")

OK - Test 4.4 - _recover_flow_read_state normal parquet/feather


### Test 4.5 - `_recover_flow_read_state` degradado bajo `strict=False`

Qué prueba:

- regeneración de `dataset_id`;
- degradación de `artifact_id` a `None`;
- `aggregation_spec = {}`;
- `provenance = {}`;
- `metadata = {}`;
- issues de recovery correspondientes.

In [16]:
payload = make_sidecar_payload(
    storage_format="feather",
    include_flow_to_trips=False,
)

payload["dataset_id"] = ""
payload["artifact_id"] = None
payload["aggregation_spec"] = None
payload["provenance"] = None
payload["metadata"] = None

issues = []

state = _recover_flow_read_state(
    payload,
    strict=False,
    issues=issues,
    destination_path=Path("/tmp/fake_degraded_read.golondrina"),
)

assert state["storage_format"] == "feather"
assert isinstance(state["dataset_id"], str)
assert state["dataset_id"].startswith("dset_")
assert state["artifact_id"] is None
assert state["aggregation_spec"] == {}
assert state["provenance"] == {}
assert state["metadata"] == {}

assert_issue_present(
    issues,
    "READ_FLOWS.METADATA.DATASET_ID_REGENERATED",
)
assert_issue_present(
    issues,
    "READ_FLOWS.METADATA.ARTIFACT_ID_SET_NONE",
)
assert_issue_present(
    issues,
    "READ_FLOWS.SIDECAR.AGGREGATION_SPEC_DEFAULTED",
)
assert_issue_present(
    issues,
    "READ_FLOWS.SIDECAR.PROVENANCE_DEFAULTED",
)
assert_issue_present(
    issues,
    "READ_FLOWS.SIDECAR.METADATA_DEFAULTED",
)

show_ok("Test 4.5 - _recover_flow_read_state degraded strict=False")

OK - Test 4.5 - _recover_flow_read_state degraded strict=False


### Test 4.6 - `_recover_flow_read_state` fatal por backend no soportado

Qué prueba: `storage.format` inválido impide reconstrucción formal.

In [18]:
payload = make_sidecar_payload(
    storage_format="feather",
    include_flow_to_trips=True,
)

payload["storage"]["format"] = "csv"

issues = []

try:
    _recover_flow_read_state(
        payload,
        strict=False,
        issues=issues,
        destination_path=Path("/tmp/fake_invalid_storage.golondrina"),
    )
    raise AssertionError("Debió fallar por storage.format no soportado")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.STORAGE.UNSUPPORTED_FORMAT",
    )

show_ok("Test 4.6 - _recover_flow_read_state unsupported storage format")

OK - Test 4.6 - _recover_flow_read_state unsupported storage format


### Test 4.7 - `_recover_flow_read_state` fatal bajo `strict=True` por identidad inválida

Qué prueba: una identidad lógica inválida no se recupera degradadamente cuando `strict=True`.

In [19]:
payload = make_sidecar_payload(
    storage_format="parquet",
    include_flow_to_trips=False,
)

payload["dataset_id"] = ""

issues = []

try:
    _recover_flow_read_state(
        payload,
        strict=True,
        issues=issues,
        destination_path=Path("/tmp/fake_strict_invalid_dataset_id.golondrina"),
    )
    raise AssertionError("Debió fallar por dataset_id inválido con strict=True")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.SIDECAR.INVALID_TOP_LEVEL",
    )

show_ok("Test 4.7 - _recover_flow_read_state strict fatal invalid dataset_id")

OK - Test 4.7 - _recover_flow_read_state strict fatal invalid dataset_id


## Bloque 5. Resolución de rutas físicas desde sidecar

Este bloque prueba la parte más importante que cambió al pasar de un diseño solo-Parquet a uno backend-aware:

- la lectura ya no asume rutas fijas;
- el archivo principal se resuelve desde `storage.format` + `files.data`;
- el auxiliar se resuelve desde `storage.format` + `files.flow_to_trips`;
- incoherencias entre backend y nombres declarados deben abortar.

### Test 5.1 - `_resolve_flows_data_path_from_sidecar` happy path para Parquet y Feather

Qué prueba:

- ruta `flows.parquet` cuando el sidecar declara Parquet;
- ruta `flows.feather` cuando el sidecar declara Feather;
- ausencia de issues.

In [20]:
for storage_format in ["parquet", "feather"]:
    case_dir = make_case_dir(f"case_05_01_data_path_{storage_format}")

    artifact = materialize_minimal_formal_flow_artifact(
        case_dir / "artifact.golondrina",
        storage_format=storage_format,
        with_aux=True,
    )

    root = artifact["paths"].root_dir
    payload = artifact["payload"]

    issues = []

    data_path = _resolve_flows_data_path_from_sidecar(
        root,
        payload,
        storage_format=storage_format,
        strict=False,
        issues=issues,
    )

    assert data_path == artifact["data_path"]
    assert data_path.exists()
    assert issues == []

show_ok("Test 5.1 - _resolve_flows_data_path_from_sidecar parquet/feather")

OK - Test 5.1 - _resolve_flows_data_path_from_sidecar parquet/feather


### Test 5.2 - `_resolve_flows_data_path_from_sidecar` fatal por mismatch backend/filename

Qué prueba: si el sidecar declara backend Feather pero `files.data="flows.parquet"`,
la ruta principal no es coherente y debe abortar.

In [21]:
case_dir = make_case_dir("case_05_02_data_path_mismatch")
root = case_dir / "artifact.golondrina"
root.mkdir(parents=True, exist_ok=True)

payload = make_sidecar_payload(
    storage_format="feather",
    include_flow_to_trips=False,
)
payload["files"]["data"] = "flows.parquet"

issues = []

try:
    _resolve_flows_data_path_from_sidecar(
        root,
        payload,
        storage_format="feather",
        strict=False,
        issues=issues,
    )
    raise AssertionError("Debió fallar por mismatch entre backend y files.data")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.LAYOUT.MISSING_DATA_FILE",
    )

show_ok("Test 5.2 - _resolve_flows_data_path_from_sidecar mismatch")

OK - Test 5.2 - _resolve_flows_data_path_from_sidecar mismatch



### Test 5.3 - `_resolve_flow_to_trips_path_from_sidecar` cuando no se solicita auxiliar

Qué prueba:

- si `requested=False`, el helper no valida el archivo declarado;
- retorna la ruta backend-aware esperada del auxiliar, aunque luego no se vaya a leer.

In [23]:
case_dir = make_case_dir("case_05_03_aux_path_not_requested")
root = case_dir / "artifact.golondrina"
root.mkdir(parents=True, exist_ok=True)

payload = make_sidecar_payload(
    storage_format="feather",
    include_flow_to_trips=False,
)

issues = []

aux_path = _resolve_flow_to_trips_path_from_sidecar(
    root,
    payload,
    storage_format="feather",
    requested=False,
    strict=False,
    issues=issues,
)

assert aux_path == root / "flow_to_trips.feather"
assert issues == []

show_ok("Test 5.3 - _resolve_flow_to_trips_path_from_sidecar requested=False")

OK - Test 5.3 - _resolve_flow_to_trips_path_from_sidecar requested=False


### Test 5.4 - `_resolve_flow_to_trips_path_from_sidecar` happy path solicitado

Qué prueba:

- con `requested=True` y sidecar coherente, se resuelve la ruta declarada;
- funciona tanto en Parquet como en Feather.

In [24]:
for storage_format in ["parquet", "feather"]:
    case_dir = make_case_dir(f"case_05_04_aux_path_{storage_format}")
    root = case_dir / "artifact.golondrina"
    root.mkdir(parents=True, exist_ok=True)

    payload = make_sidecar_payload(
        storage_format=storage_format,
        include_flow_to_trips=True,
    )

    expected_aux_name = _flow_to_trips_filename_for_storage(storage_format)

    issues = []

    aux_path = _resolve_flow_to_trips_path_from_sidecar(
        root,
        payload,
        storage_format=storage_format,
        requested=True,
        strict=False,
        issues=issues,
    )

    assert aux_path == root / expected_aux_name
    assert issues == []

show_ok("Test 5.4 - _resolve_flow_to_trips_path_from_sidecar happy parquet/feather")

OK - Test 5.4 - _resolve_flow_to_trips_path_from_sidecar happy parquet/feather


### Test 5.5 - `_resolve_flow_to_trips_path_from_sidecar` fatal por mismatch backend/filename

Qué prueba: si el sidecar declara backend Feather pero `files.flow_to_trips="flow_to_trips.parquet"`,
la ruta auxiliar es inconsistente y debe abortar.

In [25]:
case_dir = make_case_dir("case_05_05_aux_path_mismatch")
root = case_dir / "artifact.golondrina"
root.mkdir(parents=True, exist_ok=True)

payload = make_sidecar_payload(
    storage_format="feather",
    include_flow_to_trips=True,
)
payload["files"]["flow_to_trips"] = "flow_to_trips.parquet"

issues = []

try:
    _resolve_flow_to_trips_path_from_sidecar(
        root,
        payload,
        storage_format="feather",
        requested=True,
        strict=False,
        issues=issues,
    )
    raise AssertionError("Debió fallar por mismatch de flow_to_trips")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.IO.FLOW_TO_TRIPS_READ_FAILED",
    )

show_ok("Test 5.5 - _resolve_flow_to_trips_path_from_sidecar mismatch")

OK - Test 5.5 - _resolve_flow_to_trips_path_from_sidecar mismatch


## Bloque 6. Lectura de tablas físicas

Este bloque prueba los loaders reales que usa OP-11 una vez resueltas las rutas:

- tabla principal `flows`;
- auxiliar opcional `flow_to_trips`;
- dispatch correcto según backend Parquet o Feather;
- degradación y fatalidad de auxiliar faltante según `strict`.

### Test 6.1 - `_read_flows_table` con artefactos Parquet y Feather

Qué prueba:

- lectura real de tabla principal desde Parquet;
- lectura real de tabla principal desde Feather;
- forma mínima y columnas preservadas;
- ausencia de issues.

In [26]:
for storage_format in ["parquet", "feather"]:
    case_dir = make_case_dir(f"case_06_01_read_flows_table_{storage_format}")

    artifact = materialize_minimal_formal_flow_artifact(
        case_dir / "artifact.golondrina",
        storage_format=storage_format,
        with_aux=True,
    )

    issues = []

    flows_df = _read_flows_table(
        artifact["data_path"],
        storage_format=storage_format,
        issues=issues,
        destination_path=artifact["paths"].root_dir,
    )

    assert len(flows_df) == len(make_flows_df())
    assert list(flows_df.columns) == list(make_flows_df().columns)
    assert issues == []

show_ok("Test 6.1 - _read_flows_table parquet/feather")

OK - Test 6.1 - _read_flows_table parquet/feather


### Test 6.2 - `_read_optional_flow_to_trips` cuando no se solicita

Qué prueba:

- no intenta cargar;
- no emite issues;
- retorna `None`, `loaded=False`, sin archivos leídos.

In [27]:
case_dir = make_case_dir("case_06_02_aux_not_requested")
aux_path = case_dir / "flow_to_trips.feather"

issues = []

df_aux, loaded, files_read, n_rows = _read_optional_flow_to_trips(
    aux_path,
    requested=False,
    strict=False,
    storage_format="feather",
    issues=issues,
    destination_path=case_dir,
)

assert df_aux is None
assert loaded is False
assert files_read == []
assert n_rows is None
assert issues == []

show_ok("Test 6.2 - _read_optional_flow_to_trips requested=False")

OK - Test 6.2 - _read_optional_flow_to_trips requested=False


### Test 6.3 - `_read_optional_flow_to_trips` faltante bajo `strict=False`

Qué prueba:

- se solicitó auxiliar;
- el archivo no existe;
- la lectura se degrada a `None`;
- se emite warning recuperable.

In [28]:
case_dir = make_case_dir("case_06_03_aux_missing_strict_false")
aux_path = case_dir / "flow_to_trips.parquet"

issues = []

df_aux, loaded, files_read, n_rows = _read_optional_flow_to_trips(
    aux_path,
    requested=True,
    strict=False,
    storage_format="parquet",
    issues=issues,
    destination_path=case_dir,
)

assert df_aux is None
assert loaded is False
assert files_read == []
assert n_rows is None

assert_issue_present(
    issues,
    "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING",
)

show_ok("Test 6.3 - _read_optional_flow_to_trips missing strict=False")

OK - Test 6.3 - _read_optional_flow_to_trips missing strict=False


### Test 6.4 - `_read_optional_flow_to_trips` faltante bajo `strict=True`

Qué prueba:

- si el auxiliar fue pedido explícitamente y falta;
- con `strict=True` el helper aborta.

In [29]:
case_dir = make_case_dir("case_06_04_aux_missing_strict_true")
aux_path = case_dir / "flow_to_trips.feather"

issues = []

try:
    _read_optional_flow_to_trips(
        aux_path,
        requested=True,
        strict=True,
        storage_format="feather",
        issues=issues,
        destination_path=case_dir,
    )
    raise AssertionError("Debió fallar por auxiliar faltante con strict=True")
except ExportError:
    assert_issue_present(
        issues,
        "READ_FLOWS.IO.FLOW_TO_TRIPS_READ_FAILED",
    )

show_ok("Test 6.4 - _read_optional_flow_to_trips missing strict=True")

OK - Test 6.4 - _read_optional_flow_to_trips missing strict=True


### Test 6.5 - `_read_optional_flow_to_trips` con auxiliares reales Parquet y Feather

Qué prueba:

- carga real de `flow_to_trips.parquet`;
- carga real de `flow_to_trips.feather`;
- `loaded=True`;
- conteo de filas y `files_read` coherentes;
- ausencia de issues.

In [30]:
for storage_format in ["parquet", "feather"]:
    case_dir = make_case_dir(f"case_06_05_read_aux_{storage_format}")

    artifact = materialize_minimal_formal_flow_artifact(
        case_dir / "artifact.golondrina",
        storage_format=storage_format,
        with_aux=True,
    )

    issues = []

    flow_to_trips_df, loaded, files_read, n_rows = _read_optional_flow_to_trips(
        artifact["aux_path"],
        requested=True,
        strict=False,
        storage_format=storage_format,
        issues=issues,
        destination_path=artifact["paths"].root_dir,
    )

    expected_aux_name = _flow_to_trips_filename_for_storage(storage_format)

    assert loaded is True
    assert n_rows == len(make_flow_to_trips_df())
    assert files_read == [expected_aux_name]
    assert list(flow_to_trips_df.columns) == list(make_flow_to_trips_df().columns)
    assert issues == []

show_ok("Test 6.5 - _read_optional_flow_to_trips parquet/feather")

OK - Test 6.5 - _read_optional_flow_to_trips parquet/feather


## Bloque 7. Summary de lectura

### Test 7.1 - `_build_read_summary`

Qué prueba:

- summary mínimo estable de OP-11;
- conteo de flows;
- número de columnas;
- estado de carga del auxiliar;
- lista de archivos leídos;
- identidades recuperadas.

In [31]:
summary = _build_read_summary(
    flows_df=make_flows_df(),
    flow_to_trips_loaded=True,
    n_flow_to_trips=3,
    files_read=[
        "flows.feather",
        "flow_to_trips.feather",
        "flows.metadata.json",
    ],
    dataset_id="dset_001",
    artifact_id="art_001",
)

assert summary["n_flows"] == 3
assert summary["n_columns"] == len(make_flows_df().columns)
assert summary["flow_to_trips_loaded"] is True
assert summary["n_flow_to_trips"] == 3
assert summary["files_read"] == [
    "flows.feather",
    "flow_to_trips.feather",
    "flows.metadata.json",
]
assert summary["dataset_id"] == "dset_001"
assert summary["artifact_id"] == "art_001"

assert_json_dumpable(summary, "read_flows_summary")

show_ok("Test 7.1 - _build_read_summary")

OK - Test 7.1 - _build_read_summary
